# Results: Calibration & OOD Variance

Run after:
1. `python scripts/train.py --out checkpoints/bert_sst2`
2. `python scripts/eval_baseline.py --checkpoint checkpoints/bert_sst2 --out results/baseline_predictions.npz`
3. `python scripts/eval_mc_dropout.py --checkpoint checkpoints/bert_sst2 --out results/mc_predictions.npz`
4. `python scripts/eval_calibration.py --predictions results/mc_predictions.npz --baseline results/baseline_predictions.npz --out results/calibration.npz`

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if (Path.cwd().parent / "src").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
RESULTS_DIR = ROOT / "results"

## 1. Reliability diagram (calibration)

In [ ]:
cal = np.load(RESULTS_DIR / "calibration.npz", allow_pickle=True)
bin_accs, bin_confs, bin_counts, bins = cal["bin_accs"], cal["bin_confs"], cal["bin_counts"], cal["bins"]
ece = float(cal["ece"])

has_baseline = "baseline_ece" in cal
if has_baseline:
    bl_ece = float(cal["baseline_ece"])
    bl_bin_accs = cal["baseline_bin_accs"]
    bl_bin_confs = cal["baseline_bin_confs"]

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
width = (1.0 / len(bins)) * 0.4 if has_baseline else 1.0 / len(bins)
ax.bar(bins[:-1], bin_accs, width=width, align="edge", alpha=0.8, label=f"MC Dropout (ECE={ece:.3f})", color="C0")
if has_baseline:
    ax.bar(bins[:-1] + width, bl_bin_accs, width=width, align="edge", alpha=0.8, label=f"Baseline (ECE={bl_ece:.3f})", color="C1")
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Confidence (mean P(positive))")
ax.set_ylabel("Accuracy")
ax.set_title("Reliability diagram: Baseline vs MC Dropout")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print(f"MC Dropout ECE = {ece:.4f}")
if has_baseline:
    print(f"Baseline ECE   = {bl_ece:.4f}")

## 2. OOD variance comparison (ID vs Wikitext vs Corrupted)

In [ ]:
mc = np.load(RESULTS_DIR / "mc_predictions.npz", allow_pickle=True)
id_var = mc["id_var_positive"]
ood_var = mc["ood_wikitext_var"]
corr_var = mc["corr_var_positive"]

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.boxplot([id_var, ood_var, corr_var], labels=["ID (SST-2)", "OOD (Wikitext)", "Corrupted SST-2"])
ax.set_ylabel("Predictive variance σ²")
ax.set_title("MC Dropout variance: ID vs OOD")
plt.tight_layout()
plt.show()

print(f"ID    mean(var) = {id_var.mean():.4f}")
print(f"OOD   mean(var) = {ood_var.mean():.4f}  ratio = {ood_var.mean()/id_var.mean():.2f}x")
print(f"Corr  mean(var) = {corr_var.mean():.4f}  ratio = {corr_var.mean()/id_var.mean():.2f}x")

## 3. Trust-score visualization (variance as uncertainty signal)

High predictive variance σ² indicates model uncertainty. Below we show example sentences with lowest vs highest variance for ID (SST-2), corrupted SST-2, and OOD (Wikitext), so we can see which inputs the model is most/least confident about.

In [ ]:
from src.data import get_sst2_dataset, get_corrupted_sst2_dataset, get_wikitext103_dataset
from src import config

# Load variances (already loaded mc above)
id_var = mc["id_var_positive"]
corr_var = mc["corr_var_positive"]
ood_var = mc["ood_wikitext_var"]

# Load datasets to get sentences in same order as evaluation
sst2_test = get_sst2_dataset(config.SST2_SPLIT_TEST)
corr_ds = get_corrupted_sst2_dataset(split=config.SST2_SPLIT_TEST, prob=config.CORRUPTION_PROB, seed=config.SEED)
ood_ds = get_wikitext103_dataset(max_samples=len(ood_var), split="test")

id_sentences = [sst2_test[i]["sentence"] for i in range(len(id_var))]
corr_sentences = [corr_ds[i]["sentence"] for i in range(len(corr_var))]
ood_sentences = [ood_ds[i]["sentence"] for i in range(len(ood_var))]

In [ ]:
def show_trust_examples(sentences, variances, name, k=5, max_len=70):
    """Show k lowest- and k highest-variance examples."""
    order = np.argsort(variances)
    low_idx = order[:k]
    high_idx = order[-k:][::-1]
    print(f"--- {name}: {k} lowest variance ---")
    for i in low_idx:
        s = sentences[i][:max_len] + ("..." if len(sentences[i]) > max_len else "")
        print(f"  σ²={variances[i]:.4f}  {s}")
    print(f"--- {name}: {k} highest variance ---")
    for i in high_idx:
        s = sentences[i][:max_len] + ("..." if len(sentences[i]) > max_len else "")
        print(f"  σ²={variances[i]:.4f}  {s}")

show_trust_examples(id_sentences, id_var, "ID (SST-2 test)")
show_trust_examples(corr_sentences, corr_var, "Corrupted SST-2")
show_trust_examples(ood_sentences, ood_var, "OOD (Wikitext-103)")

In [ ]:
# Bar chart: variance (trust score) for sample of ID vs OOD examples
n_show = 15
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, vars_arr, titles, color in [
    (axes[0], id_var, "ID (SST-2)", "C0"),
    (axes[1], corr_var, "Corrupted SST-2", "C1"),
    (axes[2], ood_var, "OOD (Wikitext)", "C2"),
]:
    # Show first n_show samples
    v = vars_arr[:n_show]
    ax.bar(range(len(v)), v, color=color, alpha=0.8)
    ax.set_title(titles)
    ax.set_ylabel("Predictive variance σ²")
    ax.set_xlabel("Sample index")
plt.suptitle("Trust score (σ²) across first 15 samples per split")
plt.tight_layout()
plt.show()